In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import numpy as np
from datetime import datetime
import random
import os
import sys

## Load Data

In [2]:
tournaments_df = pd.read_csv('../Data/Staging/tournaments.csv').set_index(['Name', 'Year'])
players_df = pd.read_csv('../Data/Staging/players.csv').set_index('Name')

matches_df = pd.read_csv('../Data/Staging/matches.csv')
matches_df = matches_df.drop(matches_df.columns[[0]], axis = 1)
# Dropping Byes
matches_df = matches_df[(matches_df['Player 1'] != 'Bye') & (matches_df['Player 2'] != 'Bye')]
matches_df

,Player 1,Player 2,Winner,Tournament Name,Round,Year
0,Lleyton Hewitt,Guillermo Canas,Lleyton Hewitt,'S-Hertogenbosch,Finals,2001
1,Lleyton Hewitt,Roger Federer,Lleyton Hewitt,'S-Hertogenbosch,Semi-Finals,2001
2,Guillermo Canas,Tommy Robredo,Guillermo Canas,'S-Hertogenbosch,Semi-Finals,2001
3,Lleyton Hewitt,Gilles Elseneer,Lleyton Hewitt,'S-Hertogenbosch,Quarter-Finals,2001
4,Roger Federer,Raemon Sluiter,Roger Federer,'S-Hertogenbosch,Quarter-Finals,2001
...,...,...,...,...,...,...
92091,Alex Bolt,Lorenzo Giustino,Alex Bolt,Zhuhai,1st Round Qualifying,2023
92092,Dominik Palan,Chukang Wang,Dominik Palan,Zhuhai,1st Round Qualifying,2023
92093,Arthur Weber,Robert Strombachs,Arthur Weber,Zhuhai,1st Round Qualifying,2023
92094,Luke Saville,Stefanos Sakellaridis,Luke Saville,Zhuhai,1st Round Qualifying,2023


A little bit of data engineering to bring in Dates to players_df, makes it easier for feature engineering

In [3]:
dates_df = tournaments_df[['Start Date','End Date', 'Surface']]
matches_df = pd.merge(left = matches_df, right = dates_df, left_on = ['Tournament Name', 'Year'], right_on = ['Name', 'Year']).drop_duplicates()
matches_df

,Player 1,Player 2,Winner,Tournament Name,Round,Year,Start Date,End Date,Surface
0,Lleyton Hewitt,Guillermo Canas,Lleyton Hewitt,'S-Hertogenbosch,Finals,2001,2001-06-18,2001-06-24,Grass
1,Lleyton Hewitt,Roger Federer,Lleyton Hewitt,'S-Hertogenbosch,Semi-Finals,2001,2001-06-18,2001-06-24,Grass
2,Guillermo Canas,Tommy Robredo,Guillermo Canas,'S-Hertogenbosch,Semi-Finals,2001,2001-06-18,2001-06-24,Grass
3,Lleyton Hewitt,Gilles Elseneer,Lleyton Hewitt,'S-Hertogenbosch,Quarter-Finals,2001,2001-06-18,2001-06-24,Grass
4,Roger Federer,Raemon Sluiter,Roger Federer,'S-Hertogenbosch,Quarter-Finals,2001,2001-06-18,2001-06-24,Grass
...,...,...,...,...,...,...,...,...,...
92182,Alex Bolt,Lorenzo Giustino,Alex Bolt,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard
92183,Dominik Palan,Chukang Wang,Dominik Palan,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard
92184,Arthur Weber,Robert Strombachs,Arthur Weber,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard
92185,Luke Saville,Stefanos Sakellaridis,Luke Saville,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard


## Feature Engineering 

### Calculating H2H

In [4]:
# TODO convert to proportions
df = matches_df.copy()

df['Start Date'] = pd.to_datetime(df['Start Date'])
df['_row_id'] = np.arange(len(df))

a = df['Player 1'].values
b = df['Player 2'].values
df['_pair'] = np.where(a <= b, a + '||' + b, b + '||' + a)

long = pd.DataFrame({
    '_pair'      : np.r_[df['_pair'].values,      df['_pair'].values],
    '_row_id'    : np.r_[df['_row_id'].values,    df['_row_id'].values],
    'Start Date' : np.r_[df['Start Date'].values, df['Start Date'].values],
    'player'     : np.r_[df['Player 1'].values,   df['Player 2'].values],
    'won'        : np.r_[
        (df['Winner'] == df['Player 1']).astype(int).values,
        (df['Winner'] == df['Player 2']).astype(int).values
    ],
    'pos'        : np.r_[np.ones(len(df), dtype=int),
                         np.full(len(df), 2, dtype=int)]
})

long = long.sort_values(['_pair', 'Start Date', '_row_id'])

long['cum_wins'] = long.groupby(['_pair', 'player'])['won'].cumsum()
long['prev_wins'] = long.groupby(['_pair', 'player'])['cum_wins'].shift(fill_value=0)

prev = (long.pivot(index='_row_id', columns='pos', values='prev_wins')
            .rename(columns={1:'Player 1 Previous Wins', 2:'Player 2 Previous Wins'}))

matches_df[['Player 1 Previous Wins', 'Player 2 Previous Wins']] = \
    prev.loc[df['_row_id']].values

matches_df.drop(columns=['_row_id','_pair'], errors='ignore', inplace=True)
matches_df


,Player 1,Player 2,Winner,Tournament Name,Round,Year,Start Date,End Date,Surface,Player 1 Previous Wins,Player 2 Previous Wins
0,Lleyton Hewitt,Guillermo Canas,Lleyton Hewitt,'S-Hertogenbosch,Finals,2001,2001-06-18,2001-06-24,Grass,1,0
1,Lleyton Hewitt,Roger Federer,Lleyton Hewitt,'S-Hertogenbosch,Semi-Finals,2001,2001-06-18,2001-06-24,Grass,0,0
2,Guillermo Canas,Tommy Robredo,Guillermo Canas,'S-Hertogenbosch,Semi-Finals,2001,2001-06-18,2001-06-24,Grass,1,0
3,Lleyton Hewitt,Gilles Elseneer,Lleyton Hewitt,'S-Hertogenbosch,Quarter-Finals,2001,2001-06-18,2001-06-24,Grass,0,0
4,Roger Federer,Raemon Sluiter,Roger Federer,'S-Hertogenbosch,Quarter-Finals,2001,2001-06-18,2001-06-24,Grass,0,0
...,...,...,...,...,...,...,...,...,...,...,...
92182,Alex Bolt,Lorenzo Giustino,Alex Bolt,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard,0,0
92183,Dominik Palan,Chukang Wang,Dominik Palan,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard,0,0
92184,Arthur Weber,Robert Strombachs,Arthur Weber,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard,0,0
92185,Luke Saville,Stefanos Sakellaridis,Luke Saville,Zhuhai,1st Round Qualifying,2023,2023-09-20,2023-09-26,Hard,0,0


### Calculating Player Power Index
Percentage of wins in the last 10 matches played + tournament wins

In [ ]:
# --- Vectorized "previous 10 wins before this match" for each player ---
# TODO: check accuracy and also include tournament matches and convert to proportion

# 0) Copy, ensure datetime, sort by time
matches_df = matches_df.copy()
matches_df['Start Date'] = pd.to_datetime(matches_df['Start Date'])
matches_df = matches_df.sort_values(['Start Date']).reset_index(drop=True)

# 1) Stable id to merge back later
matches_df['match_id'] = np.arange(len(matches_df), dtype=np.int64)

# 2) Long form: one row per (match, player)
p1 = matches_df.rename(columns={'Player 1': 'player'})[['match_id','player','Winner','Start Date']].assign(
    is_win=lambda d: (d['Winner'] == d['player']).astype('int8'),
    side='P1'
)
p2 = matches_df.rename(columns={'Player 2': 'player'})[['match_id','player','Winner','Start Date']].assign(
    is_win=lambda d: (d['Winner'] == d['player']).astype('int8'),
    side='P2'
)
long = pd.concat([p1, p2], ignore_index=True)

# 3) Sort within player, reset index (important for clean assignment)
long = long.sort_values(['player', 'Start Date', 'match_id']).reset_index(drop=True)

# 4) Rolling wins over the PREVIOUS 10 matches (exclude current via shift)
prev = (
    long.groupby('player', sort=False, group_keys=False)
        .apply(lambda g: g['is_win'].shift(1).rolling(window=10, min_periods=1).sum())
)

long['prev10_wins'] = prev.fillna(0).astype('int16')

# 5) Map back to wide format
p1_prev = long.loc[long['side'] == 'P1', ['match_id', 'prev10_wins']] \
              .rename(columns={'prev10_wins': 'P1 Last 10 Matches'})
p2_prev = long.loc[long['side'] == 'P2', ['match_id', 'prev10_wins']] \
              .rename(columns={'prev10_wins': 'P2 Last 10 Matches'})

matches_df = (matches_df
              .merge(p1_prev, on='match_id', how='left')
              .merge(p2_prev, on='match_id', how='left')
              .drop(columns=['match_id']))

# -> matches_df now includes:
#    'P1 Last 10 Matches' and 'P2 Last 10 Matches' (int16), computed fast & correctly.
matches_df

C:\Users\aman0\AppData\Local\Temp\ipykernel_21436\1888173355.py:29: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g['is_win'].shift(1).rolling(window=10, min_periods=1).sum())


,Player 1,Player 2,Winner,Tournament Name,Round,Year,Start Date,End Date,Surface,Player 1 Previous Wins,Player 2 Previous Wins,P1 Last 10 Matches_x,P2 Last 10 Matches_x,P1 Last 10 Matches_y,P2 Last 10 Matches_y
0,Tommy Haas,Nicolas Massu,Tommy Haas,Adelaide,Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,0,0
1,Ivan Ljubicic,Andrei Pavel,Ivan Ljubicic,Adelaide,Round of 16,2001,2001-01-01,2001-01-07,Hard,0,0,1,0,0,0
2,Tommy Haas,Xavier Malisse,Tommy Haas,Adelaide,Round of 16,2001,2001-01-01,2001-01-07,Hard,0,0,1,0,1,0
3,Nicolas Massu,Arnaud Clement,Nicolas Massu,Adelaide,Round of 16,2001,2001-01-01,2001-01-07,Hard,0,0,1,0,0,0
4,Jeff Tarango,Michael Kohlmann,Jeff Tarango,Adelaide,Round of 32,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92182,Aleksandar Kovacevic,Federico Gaio,Aleksandar Kovacevic,Brisbane,1st Round Qualifying,2024,2024-12-31,2024-01-07,Hard,0,0,2,4,2,4
92183,Diego Schwartzman,Jacob Bradshaw,Diego Schwartzman,Brisbane,1st Round Qualifying,2024,2024-12-31,2024-01-07,Hard,0,0,5,0,6,0
92184,James Duckworth,Philip Sekulic,James Duckworth,Brisbane,1st Round Qualifying,2024,2024-12-31,2024-01-07,Hard,0,0,4,5,5,5
92185,Alex Michelsen,Diego Schwartzman,Alex Michelsen,Brisbane,2nd Round Qualifying,2024,2024-12-31,2024-01-07,Hard,0,0,7,6,7,7


### Calculating Player Surface Win Rate

In [46]:
import pandas as pd
import numpy as np

# 0) Ensure datetime
matches_df['Start Date'] = pd.to_datetime(matches_df['Start Date'])

# 1) Build long once (two rows per match: one for each player)
p1 = matches_df[['Player 1','Player 2','Winner','Surface','Start Date']].copy()
p1.rename(columns={'Player 1':'player','Player 2':'opponent'}, inplace=True)

p2 = matches_df[['Player 1','Player 2','Winner','Surface','Start Date']].copy()
p2.rename(columns={'Player 2':'player','Player 1':'opponent'}, inplace=True)

long = pd.concat([p1, p2], ignore_index=True)
long['is_win'] = (long['Winner'] == long['player']).astype('int8')

# 2) Collapse to per-(player, surface, date) tallies
per_date = (
    long.groupby(['player','Surface','Start Date'], as_index=False)
        .agg(matches=('player','size'), wins=('is_win','sum'))
        .sort_values(['player','Surface','Start Date'])
)

# 3) Strictly BEFORE the date (no same-day leakage): shifted cumsums
per_date['prior_matches'] = (
    per_date.groupby(['player','Surface'])['matches'].cumsum().shift(fill_value=0)
)
per_date['prior_wins'] = (
    per_date.groupby(['player','Surface'])['wins'].cumsum().shift(fill_value=0)
)

# 4) Build fast lookup Series keyed by (player, surface, date)
key_cols = ['player','Surface','Start Date']
prior_matches_s = per_date.set_index(key_cols)['prior_matches']
prior_wins_s    = per_date.set_index(key_cols)['prior_wins']

# 5) Map onto Player 1 / Player 2 rows directly
p1_keys = list(zip(matches_df['Player 1'], matches_df['Surface'], matches_df['Start Date']))
p2_keys = list(zip(matches_df['Player 2'], matches_df['Surface'], matches_df['Start Date']))

matches_df['P1 Surface Matches'] = pd.Series(p1_keys).map(prior_matches_s).fillna(0).astype(int)
matches_df['P1 Surface Wins']    = pd.Series(p1_keys).map(prior_wins_s).fillna(0).astype(int)
matches_df['P2 Surface Matches'] = pd.Series(p2_keys).map(prior_matches_s).fillna(0).astype(int)
matches_df['P2 Surface Wins']    = pd.Series(p2_keys).map(prior_wins_s).fillna(0).astype(int)

matches_df.head()

,Player 1,Player 2,Winner,Tournament Name,Round,Year,Start Date,End Date,Surface,Player 1 Previous Wins,Player 2 Previous Wins,P1 Last 10 Matches_x,P2 Last 10 Matches_x,P1 Last 10 Matches_y,P2 Last 10 Matches_y,P1 Surface Matches,P1 Surface Wins,P2 Surface Matches,P2 Surface Wins
0,Tommy Haas,Nicolas Massu,Tommy Haas,Adelaide,Finals,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,0,0,78,50,18,6
1,Ivan Ljubicic,Andrei Pavel,Ivan Ljubicic,Adelaide,Round of 16,2001,2001-01-01,2001-01-07,Hard,0,0,1,0,0,0,39,19,41,17
2,Tommy Haas,Xavier Malisse,Tommy Haas,Adelaide,Round of 16,2001,2001-01-01,2001-01-07,Hard,0,0,1,0,1,0,78,50,56,37
3,Nicolas Massu,Arnaud Clement,Nicolas Massu,Adelaide,Round of 16,2001,2001-01-01,2001-01-07,Hard,0,0,1,0,0,0,18,6,72,41
4,Jeff Tarango,Michael Kohlmann,Jeff Tarango,Adelaide,Round of 32,2001,2001-01-01,2001-01-07,Hard,0,0,0,0,0,0,1,0,18,7


### Adding in Player Rankings

In [36]:
matches_df.to_csv('../Data/matches.csv')